# 📘 The AI Engineer's LLM Workbook

**14 Chapters · 14 Google Colab Notebooks · Beginner to Production**

---

*© 2026 JAWNVION LLC — www.jawnvion.com — peter@jawnvion.com*

*Licensed for individual use. Do not redistribute.*

---

## What's Inside

| # | Chapter |
|---|---------|
| 01 | AI Fundamentals & Problem Framing |
| 02 | Data Science Toolkit (NumPy, Pandas, Matplotlib) |
| 03 | Neural Networks from Scratch |
| 04 | Transformers Architecture Deep Dive |
| 05 | HuggingFace & Pre-Trained Models |
| 06 | QLoRA Fine-Tuning |
| 07 | DPO Alignment Training |
| 08 | Retrieval-Augmented Generation (RAG) |
| 09 | Model Evaluation & Benchmarking |
| 10 | FastAPI Deployment |
| 11 | Monitoring & Observability |
| 12 | Security for AI Systems |
| 13 | Cost Optimization & Quantization |
| 14 | Capstone: End-to-End LLM Project |

---

> **How to use:** Click **Runtime → Run All** in Google Colab, or run cells one at a time.
> Each chapter builds on the last — complete them in order for best results.

---


# Chapter 8: RAG — Retrieval-Augmented Generation
**JAWNVION LLC — AI Training Workbook**

RAG fixes the core weakness of a fine-tuned LLM: it can't answer questions about facts it
was never trained on. Instead of retraining, we give the model a *retrieval system* — a
searchable knowledge base it can look up at inference time.

**What you'll learn:**
- Why RAG outperforms a fine-tuned model on closed-domain Q&A
- How to build a FAISS vector index from a document corpus
- How to embed queries and retrieve top-k passages
- How to combine retrieval + generation into a single RAG pipeline
- How RAG compares to a base LLM with no context (before/after demo)

In [ ]:
# — Cell 1: GPU Check ——————————————————————————————————
import torch, subprocess, sys

result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                        '--format=csv,noheader'], capture_output=True, text=True)
if result.returncode == 0:
    print('✓  GPU detected:', result.stdout.strip())
else:
    print('⚠  No GPU found — RAG retrieval runs on CPU (slower but functional)')
    print('   For the LLM generation step, Runtime → Change runtime type → T4 GPU')

print(f'   PyTorch: {torch.__version__}')

In [ ]:
# — Cell 2: Install Packages ——————————————————————————
# Pin tokenizers first to avoid Rust wheel compilation
!pip install -q "tokenizers>=0.22,<0.24"
!pip install -q -U transformers datasets accelerate
!pip install -q sentence-transformers faiss-cpu
print('✓  Packages installed')
print('   If prompted to restart runtime, do so then Run All again')

In [ ]:
# — Cell 3: Imports & Configuration ———————————————————
import torch
import numpy as np
import time
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
import faiss

# ── Constants ─────────────────────────────────────────
EMBED_MODEL    = "all-MiniLM-L6-v2"        # 80 MB, fast, strong retrieval
LLM_MODEL      = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
NUM_DOCS       = 500                        # passages to index
TOP_K          = 3                          # passages retrieved per query
MAX_CTX_TOKENS = 512                        # max tokens fed to LLM

print('✓  Config ready')
print(f'   Embedding model : {EMBED_MODEL}')
print(f'   LLM             : {LLM_MODEL}')
print(f'   Knowledge base  : {NUM_DOCS} passages')
print(f'   Top-K retrieval : {TOP_K}')

In [ ]:
# — Cell 4: Load Knowledge Base ———————————————————————
# We use SQuAD context passages as our "documents" — short Wikipedia paragraphs
# that the base LLM may not answer precisely without retrieval.

print("Loading SQuAD dataset for knowledge-base passages...")
squad = load_dataset("rajpurkar/squad", split="train")

# Deduplicate by context text — SQuAD has many questions per passage
seen, passages = set(), []
for ex in squad:
    ctx = ex["context"].strip()
    if ctx not in seen:
        seen.add(ctx)
        passages.append({
            "text": ctx,
            "title": ex["title"],
        })
    if len(passages) >= NUM_DOCS:
        break

print(f"✓  Knowledge base ready: {len(passages)} unique passages")
print()
print("Sample passage:")
print(f"  Title : {passages[0]['title']}")
print(f"  Text  : {passages[0]['text'][:200]}...")

In [ ]:
# — Cell 5: Embed Passages & Build FAISS Index ———————
# SentenceTransformer converts each passage to a 384-dim vector.
# FAISS stores those vectors for fast nearest-neighbour search.

print(f"Loading embedding model: {EMBED_MODEL}")
embedder = SentenceTransformer(EMBED_MODEL)

print(f"Embedding {len(passages)} passages...")
t0 = time.time()
corpus_texts = [p["text"] for p in passages]
corpus_embeddings = embedder.encode(
    corpus_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,   # normalise for cosine similarity via dot product
)
print(f"✓  Embeddings ready in {time.time()-t0:.1f}s  |  shape: {corpus_embeddings.shape}")

# Build a flat L2 index (exact search — fine for 500 docs)
dim = corpus_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)          # Inner Product = cosine sim (after normalisation)
index.add(corpus_embeddings.astype("float32"))
print(f"✓  FAISS index built  |  {index.ntotal} vectors  |  dim={dim}")

In [ ]:
# — Cell 6: Retrieval Function & Test ————————————————
def retrieve(query: str, k: int = TOP_K) -> list[dict]:
    """Embed query, search index, return top-k passage dicts."""
    q_emb = embedder.encode([query], normalize_embeddings=True).astype("float32")
    scores, indices = index.search(q_emb, k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "title": passages[idx]["title"],
            "text":  passages[idx]["text"],
            "score": float(score),
        })
    return results

# ── Quick test ─────────────────────────────────────────
TEST_QUERY = "Who invented the telephone?"
print(f"Query: {TEST_QUERY}")
print()
hits = retrieve(TEST_QUERY)
for i, h in enumerate(hits):
    print(f"  Rank {i+1}  (score={h['score']:.3f})  [{h['title']}]")
    print(f"    {h['text'][:150]}...")
    print()
print("✓  Retrieval working")

In [ ]:
# — Cell 7: Load TinyLlama LLM ————————————————————————
# We use the base model (no fine-tuning) to show that RAG improves
# a stock LLM — the retrieval, not the weights, provides the knowledge.

print(f"Loading tokenizer: {LLM_MODEL}")
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

print(f"Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()

vram = torch.cuda.memory_allocated()/1e9 if torch.cuda.is_available() else 0
print(f"✓  Model loaded  |  VRAM used: {vram:.2f} GB")

In [ ]:
# — Cell 8: RAG Pipeline ——————————————————————————————
# The full pipeline:
#   1. Embed the question
#   2. Retrieve top-K passages from FAISS
#   3. Inject passages as context into the prompt
#   4. Generate the answer

def build_rag_prompt(question: str, passages: list[dict]) -> str:
    context_block = ""
    for i, p in enumerate(passages):
        context_block += f"[{i+1}] {p['title']}\n{p['text']}\n\n"
    return (
        "Read the context below and answer the question concisely.\n\n"
        f"CONTEXT:\n{context_block}"
        f"QUESTION: {question}\n\n"
        "ANSWER:"
    )

def generate(prompt: str, max_new: int = 120) -> str:
    inputs = tokenizer(prompt, return_tensors="pt",
                       truncation=True, max_length=MAX_CTX_TOKENS).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new,
            do_sample=False,
            repetition_penalty=1.3,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()

def rag_answer(question: str) -> tuple[str, list[dict]]:
    hits = retrieve(question)
    prompt = build_rag_prompt(question, hits)
    answer = generate(prompt)
    return answer, hits

def base_answer(question: str) -> str:
    prompt = f"Question: {question}\n\nAnswer:"
    return generate(prompt)

print("✓  RAG pipeline ready")
print("   Functions: rag_answer(question), base_answer(question)")

In [ ]:
# — Cell 9: Before / After Comparison ————————————————
# Show how RAG changes the answer quality on questions that require
# specific factual knowledge from the indexed corpus.

QUESTIONS = [
    "What city was Nikola Tesla born in?",
    "What is the speed of light in a vacuum?",
    "Who wrote the play Hamlet?",
]

print("=" * 70)
print("  RAG vs. Base LLM — Side-by-Side Comparison")
print("=" * 70)

for q in QUESTIONS:
    print(f"\nQUESTION: {q}")
    print("─" * 60)

    print("BASE LLM (no context):")
    base = base_answer(q)
    print(f"  {base}")

    print()
    print(f"RAG (top-{TOP_K} passages retrieved):")
    rag, hits = rag_answer(q)
    print(f"  {rag}")

    print()
    print("Retrieved passages:")
    for i, h in enumerate(hits):
        print(f"  [{i+1}] {h['title']}  (score={h['score']:.3f})")
    print()

print("=" * 70)
print("KEY INSIGHT:")
print("  The base LLM may hallucinate or be vague.")
print("  The RAG model grounds its answer in retrieved text.")
print("  Better retrieval → better answers, no retraining required.")
print("=" * 70)

In [ ]:
# — Cell 10: Save FAISS Index ————————————————————————
import os, json as _json, shutil

SAVE_DIR = "/content/rag-index"
os.makedirs(SAVE_DIR, exist_ok=True)

# Save FAISS index
faiss.write_index(index, f"{SAVE_DIR}/passages.index")

# Save passage metadata
with open(f"{SAVE_DIR}/passages.json", "w") as f:
    _json.dump(passages, f)

# Zip for download
shutil.make_archive("/content/rag-index-backup", "zip", SAVE_DIR)

print(f"✓  FAISS index saved to {SAVE_DIR}/passages.index")
print(f"   Passage metadata  : {SAVE_DIR}/passages.json")
print(f"   Backup zip        : /content/rag-index-backup.zip")
print()
print("To reload in a new session:")
print("  index = faiss.read_index('passages.index')")
print("  passages = json.load(open('passages.json'))")

## Chapter 8 Complete ✓

**What happened:**
- Loaded 500 Wikipedia passages (SQuAD) as a searchable knowledge base
- Built 384-dim sentence embeddings with `all-MiniLM-L6-v2`
- Indexed embeddings in FAISS (flat inner-product index — exact cosine search)
- Built a RAG pipeline: embed query → retrieve top-3 → inject context → generate
- Compared base LLM (no context) vs. RAG-augmented on factual questions

**Key RAG concepts demonstrated:**
- `SentenceTransformer` embeds both documents and queries into the same vector space
- `faiss.IndexFlatIP` does exact cosine similarity search (fast enough for <10k docs)
- Context injection is a prompt-engineering step — no model retraining needed
- Retrieval quality determines answer quality; the LLM is the reader, not the memory

**When to use RAG vs. fine-tuning:**
| | RAG | Fine-tuning |
|---|---|---|
| Knowledge source | External, updatable | Baked into weights |
| Add new facts | Re-index (minutes) | Retrain (hours/days) |
| Precise Q&A | ✓ Strong | ✗ Prone to hallucination |
| Style/tone adaptation | ✗ | ✓ Strong |

**Next: Chapter 9 — Evaluation & Red-Teaming**